In [1]:
import anndata as ad
import shutil
import sys
from pathlib import Path

import celltypist
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas.core.indexes.base as pandas_indexes_base
import scanpy as sc
import seaborn as sns
from celltypist import models
from scipy.sparse import csr_matrix

# Configuring scanpy's settings for outfits and visualization
sc.settings.verbosity = 0

c:\Users\asd123cheese\AppData\Local\Programs\Python\Python314\Lib\site-packages\celltypist\classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


In [2]:
adata = ad.read_h5ad("../data_interim/dimensionality_reduced.h5ad")

In [3]:
for i, ct in enumerate(adata.obs["cell_ontology_class"].unique(), 1):
    print(i, ct)

1 endothelial cell
2 B cell
3 mesenchymal stem cell of adipose
4 T cell
5 myeloid cell
6 erythroblast
7 epithelial cell
8 lymphocyte
9 fibroblast of cardiac tissue
10 leukocyte
11 endocardial cell
12 smooth muscle cell
13 cardiomyocyte
14 endothelial cell of coronary artery
15 cardiac neuron
16 erythrocyte
17 mast cell
18 kidney proximal convoluted tubule epithelial cell
19 macrophage
20 kidney loop of Henle thick ascending limb epithelial cell
21 kidney distal convoluted tubule epithelial cell
22 kidney loop of Henle ascending limb epithelial cell
23 kidney collecting duct principal cell
24 fenestrated cell
25 epithelial cell of proximal tubule
26 brush cell
27 podocyte
28 kidney cortex artery cell
29 NK cell
30 kidney mesangial cell
31 kidney capillary endothelial cell
32 fibroblast
33 plasma cell
34 kidney proximal straight tubule epithelial cell
35 kidney collecting duct epithelial cell
36 kidney cell
37 hepatocyte
38 endothelial cell of hepatic sinusoid
39 Kupffer cell
40 hepatic 

In [4]:
celltype_dict = {
# endothelial
"endothelial cell": "endothelial",
"endothelial cell of coronary artery": "endothelial",
"endothelial cell of hepatic sinusoid": "endothelial",
"kidney capillary endothelial cell": "endothelial",
"vein endothelial cell": "endothelial",
"endothelial cell of lymphatic vessel": "endothelial",
"fenestrated cell": "endothelial",
"endocardial cell": "endothelial",
"kidney cortex artery cell": "endothelial",
# myeloid/macrophage
"myeloid cell": "myeloid/macrophage",
"macrophage": "myeloid/macrophage",
"lung macrophage": "myeloid/macrophage",
"alveolar macrophage": "myeloid/macrophage",
"Kupffer cell": "myeloid/macrophage",
"monocyte": "myeloid/macrophage",
"classical monocyte": "myeloid/macrophage",
"non-classical monocyte": "myeloid/macrophage",
"intermediate monocyte": "myeloid/macrophage",
"promonocyte": "myeloid/macrophage",
"myeloid leukocyte": "myeloid/macrophage",
"myeloid dendritic cell": "myeloid/macrophage",
"macrophage dendritic cell progenitor": "myeloid/macrophage",
"dendritic cell": "myeloid/macrophage",
"plasmacytoid dendritic cell": "myeloid/macrophage",
"granulocyte": "myeloid/macrophage",
"neutrophil": "myeloid/macrophage",
"basophil": "myeloid/macrophage",
"mast cell": "myeloid/macrophage",
"granulocytopoietic cell": "myeloid/macrophage",
# fibroblast/stromal
"fibroblast": "fibroblast/stromal",
"fibroblast of cardiac tissue": "fibroblast/stromal",
"fibroblast of lung": "fibroblast/stromal",
"pulmonary interstitial fibroblast": "fibroblast/stromal",
"mesenchymal stem cell of adipose": "fibroblast/stromal",
"hepatic stellate cell": "fibroblast/stromal",
"kidney mesangial cell": "fibroblast/stromal",
"adventitial cell": "fibroblast/stromal",
"pericyte cell": "fibroblast/stromal",
# immune lymphoid
"B cell": "lymphoid",
"T cell": "lymphoid",
"CD4-positive, alpha-beta T cell": "lymphoid",
"CD8-positive, alpha-beta T cell": "lymphoid",
"naive T cell": "lymphoid",
"regulatory T cell": "lymphoid",
"NK cell": "lymphoid",
"mature NK T cell": "lymphoid",
"immature NKT cell": "lymphoid",
"lymphocyte": "lymphoid",
"plasma cell": "lymphoid",
"naive B cell": "lymphoid",
"precursor B cell": "lymphoid",
"late pro-B cell": "lymphoid",
"immature B cell": "lymphoid",
# epithelial
"epithelial cell": "epithelial",
"kidney proximal convoluted tubule epithelial cell": "epithelial",
"kidney loop of Henle thick ascending limb epithelial cell": "epithelial",
"kidney distal convoluted tubule epithelial cell": "epithelial",
"kidney loop of Henle ascending limb epithelial cell": "epithelial",
"kidney collecting duct principal cell": "epithelial",
"epithelial cell of proximal tubule": "epithelial",
"brush cell": "epithelial",
"podocyte": "epithelial",
"kidney proximal straight tubule epithelial cell": "epithelial",
"kidney collecting duct epithelial cell": "epithelial",
"kidney cell": "epithelial",
"hepatocyte": "epithelial",
"duct epithelial cell": "epithelial",
"type II pneumocyte": "epithelial",
"ciliated columnar cell of tracheobronchial tree": "epithelial",
"club cell of bronchiole": "epithelial",
"lung neuroendocrine cell": "epithelial",
# muscle
"smooth muscle cell": "muscle",
"smooth muscle cell of the pulmonary artery": "muscle",
"bronchial smooth muscle cell": "muscle",
"cardiomyocyte": "muscle",
# hematopoietic/erythroid
"erythroblast": "hematopoietic/erythroid",
"proerythroblast": "hematopoietic/erythroid",
"erythroid progenitor": "hematopoietic/erythroid",
"megakaryocyte-erythroid progenitor cell": "hematopoietic/erythroid",
"erythrocyte": "hematopoietic/erythroid",
"hematopoietic precursor cell": "hematopoietic/erythroid",
# other
"cardiac neuron": "other",
"leukocyte": "other"
}

In [5]:
adata.obs["major_cell_type"] = adata.obs["cell_ontology_class"].map(celltype_dict).fillna("unmapped")
n_unmapped = int((adata.obs["major_cell_type"] == "unmapped").sum())
print(f"未映射细胞数: {n_unmapped} / {len(adata.obs)}")

未映射细胞数: 0 / 140200


In [6]:
cell_count = (adata.obs.groupby(["mouse.id","tissue","major_cell_type"]).size().reset_index(name="cell_number"))

cell_count.head()

,mouse.id,tissue,major_cell_type,cell_number
0,1-M-62,Fat,endothelial,0
1,1-M-62,Fat,epithelial,0
2,1-M-62,Fat,fibroblast/stromal,0
3,1-M-62,Fat,hematopoietic/erythroid,0
4,1-M-62,Fat,lymphoid,0


In [7]:
cell_count.to_csv("../results/qc/final_cohort_matrix.csv",index=False)

In [8]:
coverage = (cell_count.pivot_table(index=["tissue","major_cell_type"],columns="mouse.id",values="cell_number",fill_value=0))

coverage.head()

mouse.id                        1-M-62  1-M-63  3-F-56  3-F-57  3-M-5/6  \
tissue major_cell_type                                                    
Fat    endothelial                 0.0     0.0     0.0     0.0      0.0   
       epithelial                  0.0     0.0     0.0     0.0      0.0   
       fibroblast/stromal          0.0     0.0     0.0     0.0      0.0   
       hematopoietic/erythroid     0.0     0.0     0.0     0.0      0.0   
       lymphoid                    0.0     0.0     0.0     0.0      0.0   

mouse.id                        3-M-7/8  3-M-8  3-M-8/9  3-M-9  18-F-50  ...  \
tissue major_cell_type                                                   ...   
Fat    endothelial                  0.0    0.0      0.0    0.0    288.0  ...   
       epithelial                   0.0    0.0      0.0    0.0     50.0  ...   
       fibroblast/stromal           0.0    0.0      0.0    0.0    320.0  ...   
       hematopoietic/erythroid      0.0    0.0      0.0    0.0      7.0  ...   
       lymphoid                     0.0    0.0      0.0    0.0    517.0  ...   

mouse.id                        21-F-54  21-F-55  24-M-58  24-M-59  24-M-60  \
tissue major_cell_type                                                        
Fat    endothelial                 71.0    117.0      0.0      0.0      0.0   
       epithelial                  31.0     11.0      0.0      0.0      0.0   
       fibroblast/stromal         123.0    126.0      0.0      0.0      0.0   
       hematopoietic/erythroid      7.0      6.0      0.0      0.0      0.0   
       lymphoid                    74.0     62.0      0.0      0.0      0.0   

mouse.id                        24-M-61  30-M-2  30-M-3  30-M-4  30-M-5  
tissue major_cell_type                                                   
Fat    endothelial                  0.0     0.0     0.0     0.0   645.0  
       epithelial                   0.0     0.0     0.0     0.0     5.0  
       fibroblast/stromal           0.0     0.0     0.0     0.0   659.0  
       hematopoietic/erythroid      0.0     0.0     0.0     0.0    18.0  
       lymphoid                     0.0     0.0     0.0     0.0   475.0  

[5 rows x 23 columns]

In [9]:
coverage_binary = coverage >= 30

coverage_summary = (coverage_binary.groupby(level=["tissue","major_cell_type"]).mean())

coverage_summary

mouse.id                                 1-M-62  1-M-63  3-F-56  3-F-57  \
tissue          major_cell_type                                           
Fat             endothelial                 0.0     0.0     0.0     0.0   
                epithelial                  0.0     0.0     0.0     0.0   
                fibroblast/stromal          0.0     0.0     0.0     0.0   
                hematopoietic/erythroid     0.0     0.0     0.0     0.0   
                lymphoid                    0.0     0.0     0.0     0.0   
                muscle                      0.0     0.0     0.0     0.0   
                myeloid/macrophage          0.0     0.0     0.0     0.0   
                other                       0.0     0.0     0.0     0.0   
Heart_and_Aorta endothelial                 0.0     1.0     1.0     0.0   
                epithelial                  0.0     0.0     0.0     0.0   
                fibroblast/stromal          0.0     1.0     1.0     0.0   
                hematopoietic/erythroid     0.0     0.0     0.0     0.0   
                lymphoid                    0.0     0.0     0.0     0.0   
                muscle                      0.0     1.0     1.0     0.0   
                myeloid/macrophage          0.0     0.0     0.0     0.0   
                other                       0.0     1.0     1.0     0.0   
Kidney          endothelial                 1.0     1.0     0.0     1.0   
                epithelial                  1.0     1.0     0.0     1.0   
                fibroblast/stromal          1.0     1.0     0.0     1.0   
                hematopoietic/erythroid     0.0     0.0     0.0     0.0   
                lymphoid                    0.0     0.0     0.0     0.0   
                muscle                      0.0     0.0     0.0     0.0   
                myeloid/macrophage          0.0     1.0     0.0     1.0   
                other                       0.0     0.0     0.0     0.0   
Liver           endothelial                 0.0     1.0     0.0     0.0   
                epithelial                  1.0     1.0     1.0     1.0   
                fibroblast/stromal          0.0     0.0     0.0     0.0   
                hematopoietic/erythroid     0.0     0.0     0.0     0.0   
                lymphoid                    0.0     1.0     0.0     0.0   
                muscle                      0.0     0.0     0.0     0.0   
                myeloid/macrophage          0.0     1.0     0.0     0.0   
                other                       0.0     0.0     0.0     0.0   
Lung            endothelial                 0.0     0.0     0.0     0.0   
                epithelial                  0.0     0.0     0.0     0.0   
                fibroblast/stromal          1.0     1.0     0.0     1.0   
                hematopoietic/erythroid     0.0     0.0     0.0     0.0   
                lymphoid                    1.0     1.0     1.0     1.0   
                muscle                      1.0     1.0     1.0     1.0   
                myeloid/macrophage          1.0     1.0     1.0     1.0   
                other                       0.0     0.0     0.0     0.0   
spleen/marrow   endothelial                 0.0     0.0     0.0     0.0   
                epithelial                  0.0     0.0     0.0     0.0   
                fibroblast/stromal          0.0     0.0     0.0     0.0   
                hematopoietic/erythroid     1.0     1.0     1.0     1.0   
                lymphoid                    1.0     1.0     1.0     1.0   
                muscle                      0.0     0.0     0.0     0.0   
                myeloid/macrophage          1.0     1.0     1.0     1.0   
                other                       0.0     0.0     0.0     0.0   

mouse.id                                 3-M-5/6  3-M-7/8  3-M-8  3-M-8/9  \
tissue          major_cell_type                                             
Fat             endothelial                  0.0      0.0    0.0      0.0   
                ep

In [14]:
n_mouse = coverage_summary.shape[1]

coverage_table = (coverage_summary.sum(axis=1).reset_index())

coverage_table.columns = ["tissue","cell_type","mouse_pass_number"]

coverage_table["total_mouse"] = n_mouse

coverage_table

,tissue,cell_type,mouse_pass_number,total_mouse
0,Fat,endothelial,6.0,23
1,Fat,epithelial,2.0,23
2,Fat,fibroblast/stromal,6.0,23
3,Fat,hematopoietic/erythroid,1.0,23
4,Fat,lymphoid,5.0,23
5,Fat,muscle,0.0,23
6,Fat,myeloid/macrophage,6.0,23
7,Fat,other,0.0,23
8,Heart_and_Aorta,endothelial,11.0,23
9,Heart_and_Aorta,epithelial,0.0,23


In [15]:
coverage_table.to_csv("../results/tissue_celltype_coverage_statistics.csv",index=False)

In [20]:
target_cells = ["endothelial","fibroblast/stromal","lymphoid","myeloid/macrophage","epithelial"]

adata = adata[adata.obs['major_cell_type'].isin(target_cells)].copy()

In [24]:
adata.write_h5ad("../data_interim/cell_type_filtered.h5ad")

... storing 'major_cell_type' as categorical


In [28]:
adata.obs

,age,cell,cell_ontology_class,cell_ontology_id,free_annotation,method,mouse.id,n_genes,sex,subtissue,...,pct_counts_in_top_20_genes,total_counts_mt,log1p_total_counts_mt,pct_counts_mt,total_counts_ribo,log1p_total_counts_ribo,pct_counts_ribo,scDblFinder_score,scDblFinder_class,major_cell_type
index,,,,,,,,,,,,,,,,,,,,,
AAAGTAGTCTCATTCA-1-6-0-0,18m,MACA_18m_F_GAT_50_AAAGTAGTCTCATTCA,endothelial cell,NA,Cd36+,droplet,18-F-50,1240,female,Gat,...,22.140809,0.0,0.0,0.0,749.0,6.620073,23.861103,0.005353,singlet,endothelial
AACCGCGAGCCGATTT-1-6-0-0,18m,MACA_18m_F_GAT_50_AACCGCGAGCCGATTT,B cell,NA,nan,droplet,18-F-50,1309,female,Gat,...,22.680891,0.0,0.0,0.0,1708.0,7.443664,39.610390,0.019777,singlet,lymphoid
AACTCAGGTTGGTGGA-1-6-0-0,18m,MACA_18m_F_GAT_50_AACTCAGGTTGGTGGA,mesenchymal stem cell of adipose,NA,nan,droplet,18-F-50,1189,female,Gat,...,18.501873,0.0,0.0,0.0,720.0,6.580639,26.966291,0.012113,singlet,fibroblast/stromal
AACTCCCAGGCATGTG-1-6-0-0,18m,MACA_18m_F_GAT_50_AACTCCCAGGCATGTG,T cell,NA,nan,droplet,18-F-50,978,female,Gat,...,26.181004,0.0,0.0,0.0,2279.0,7.731931,56.663353,0.010406,singlet,lymphoid
AACTCTTTCTTACCTA-1-6-0-0,18m,MACA_18m_F_GAT_50_AACTCTTTCTTACCTA,endothelial cell,NA,Cd36+,droplet,18-F-50,4094,female,Gat,...,20.665484,0.0,0.0,0.0,6503.0,8.780172,28.850931,0.838873,doublet,endothelial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10X_P7_3_TTTGGTTCAGGTCTCG-1,3m,10X_P7_3_TTTGGTTCAGGTCTCG,precursor B cell,CL:0000235,nan,droplet,3-F-57,1961,female,nan,...,17.185212,0.0,0.0,0.0,1657.0,7.413367,30.325768,0.003270,singlet,lymphoid
10X_P7_3_TTTGGTTTCTGCTTGC-1,3m,10X_P7_3_TTTGGTTTCTGCTTGC,promonocyte,CL:0000559,nan,droplet,3-F-57,2061,female,nan,...,19.375476,0.0,0.0,0.0,1669.0,7.420579,25.422695,0.088842,singlet,myeloid/macrophage
10X_P7_3_TTTGTCAAGCGCTCCA-1,3m,10X_P7_3_TTTGTCAAGCGCTCCA,naive B cell,CL:0000767,nan,droplet,3-F-57,3320,female,nan,...,16.432283,0.0,0.0,0.0,5168.0,8.550434,35.801868,0.006412,singlet,lymphoid


['Fat', 'Kidney', 'Liver', 'Lung']

In [82]:
df = pd.read_csv("../results/tissue_celltype_coverage_statistics.csv")

target_tissue=[]

for a in df["tissue"].unique():
    for b in df[df["tissue"] == a]["cell_type"].unique():
        sub = df[(df["tissue"] == a) & (df["cell_type"] == b)]
        cond1 = sub["mouse_pass_number"].min() >= 3
        cond2 = (
            df[(df["tissue"] == a)
               & (df["mouse_pass_number"] >= 3)
               & (df["cell_type"].isin(target_cells))]
            ["cell_type"]
            .nunique() >= 3
        )
        if cond1 and cond2:
            target_tissue.append(a)
            break

print("正式分析组织:")
print(target_tissue)

正式分析组织:
['Fat', 'Kidney', 'Liver', 'Lung']


In [83]:
adata = adata[adata.obs['tissue'].isin(target_tissue)].copy()

In [92]:
adata.write_h5ad("../data_interim/final_filtered.h5ad")